In [ ]:
# 1. 필수 라이브러리 설치
!pip install -q transformers datasets accelerate evaluate

# 2. 구글 드라이브 연동
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

import random
from tqdm.notebook import tqdm
tqdm.pandas()

In [ ]:
# ==========================================
# 1. 데이터 로드
# ==========================================

data_path = '/content/drive/MyDrive/시냅스 팀플_2/dataset/train.csv'
df = pd.read_csv(data_path)
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==========================================
# 2. ByT5 모델 및 토크나이저 로드
# ==========================================

model_name = "google/byt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
# ==========================================
# 3. 데이터 전처리 (토큰화)
# ==========================================

def preprocess_function(examples):
    inputs = examples["input"]
    targets = examples["output"]

    # 입력과 정답 모두 max_length=256으로 꽉꽉 채워서 고정
    model_inputs = tokenizer(inputs, max_length=256, padding="max_length", truncation=True)
    labels = tokenizer(targets, max_length=256, padding="max_length", truncation=True)

    # ★ 핵심: 정답(Label)의 빈칸(pad) 토큰을 -100으로 바꿔서 모델이 계산할 때 무시하도록 강제
    labels_with_ignore_index = []
    for label in labels["input_ids"]:
        # 패딩 토큰이면 -100으로, 아니면 원래 토큰 유지
        label = [l if l != tokenizer.pad_token_id else -100 for l in label]
        labels_with_ignore_index.append(label)

    model_inputs["labels"] = labels_with_ignore_index
    return model_inputs


In [ ]:
# ==========================================
# 4. 평가 지표 (Char F1 Score) 계산 함수
# ==========================================
def calc_char_f1(pred_str, true_str):
    pred_chars = list(pred_str.replace(" ", ""))
    true_chars = list(true_str.replace(" ", ""))

    if len(pred_chars) == 0 or len(true_chars) == 0:
        return 0.0

    common_chars = set(pred_chars) & set(true_chars)
    if not common_chars:
        return 0.0

    precision = len(common_chars) / len(set(pred_chars))
    recall = len(common_chars) / len(set(true_chars))

    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    f1_scores = [calc_char_f1(p, l) for p, l in zip(decoded_preds, decoded_labels)]
    return {"char_f1": np.mean(f1_scores)}

In [ ]:
# ==========================================
# 5. 모델 학습 설정 (드라이브에 저장되도록 수정)
# ==========================================

def preprocess_function(examples):
    inputs = examples["input"]
    targets = examples["output"]

    # 변경점: padding="max_length" 옵션을 추가하여 attention_mask가 절대 누락되지 않게 고정합니다.
    model_inputs = tokenizer(inputs, max_length=256, padding="max_length", truncation=True)
    labels = tokenizer(targets, max_length=256, padding="max_length", truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
from transformers import DataCollatorForSeq2Seq
import numpy as np

# ==========================================
# 1. 필수 부품
# ==========================================
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

def calc_char_f1(pred_str, true_str):
    pred_chars = list(pred_str.replace(" ", ""))
    true_chars = list(true_str.replace(" ", ""))
    if len(pred_chars) == 0 or len(true_chars) == 0: return 0.0
    common_chars = set(pred_chars) & set(true_chars)
    if not common_chars: return 0.0
    precision = len(common_chars) / len(set(pred_chars))
    recall = len(common_chars) / len(set(true_chars))
    return 2 * (precision * recall) / (precision + recall)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    f1_scores = [calc_char_f1(p, l) for p, l in zip(decoded_preds, decoded_labels)]
    return {"char_f1": np.mean(f1_scores)}

# ==========================================
# 2. 모델 세팅
# ==========================================
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=False,
    logging_steps=50,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# ==========================================
# 3. 학습 시작
# ==========================================
print("드디어 에러를 잡았습니다. 학습을 시작합니다!!!")

# 1. 이상한 토큰 생성 방지를 위한 데이터 필터링
def filter_bad_tokens(example):
    # 라벨에 0x110000 이상의 값이 들어가는 것을 방지하기 위한 안전장치
    return all(label == -100 or 0 <= label < 1114112 for label in example["labels"])

print("데이터셋 안전 검사 중...")
tokenized_train = tokenized_train.filter(filter_bad_tokens)
tokenized_val = tokenized_val.filter(filter_bad_tokens)

# 2. 학습 시작
trainer.train()

드디어 에러를 잡았습니다. 학습을 시작합니다!!!
데이터셋 안전 검사 중...


Filter:   0%|          | 0/20272 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1127 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Char F1
1,0.025565,0.099695,0.290092


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# 모델 저장
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)
print("학습 완료 및 드라이브 저장 완료")